# CWT → Vision Encoder → Multi-Head Indicator Decoder

**Idea.** Feed a deterministic causal CWT scalogram into a small 2D CNN, and train multiple heads to reconstruct classical technical indicators (RSI, MACD, Bollinger Bands) over the full window.

**Why this is a non-trivial experiment even though indicators are deterministic functions of price:**

- The CWT representation loses pointwise amplitude (rolling-normalized), so the decoder can't just invert and apply the formula — it has to learn from the scalogram's structural features.
- A 2D CNN's inductive bias (locality, multi-scale pooling) is biased away from exact analytic inversion and toward pattern-matching across scale × time.
- The resulting shared latent is a candidate feature representation for downstream predictive heads (see returns-head stub at the bottom).

**Framework:** JAX + Flax + optax. PyTorch has no Python 3.13 + Intel macOS wheels on this machine (see `CLAUDE.md`).

**Output shapes recap:**
- Input window: `(batch, n_scales, W)` CWT coefficients → reshape to `(batch, n_scales, W, 1)` for Flax NHWC.
- Shared latent: `(batch, W, C)` — scale dim collapsed, time dim preserved.
- Per-head output: `(batch, W, n_channels_head)` full trajectory across the window.

---

In [ ]:
# Colab setup — clone repo (skip stale submodule) and pip-install workspace packages.
# This is a no-op when running locally via `uv run jupyter notebook` from the repo root.
#
# Use --no-deps so we keep Colab's preinstalled jax/flax/optax instead of letting
# ss-indicators' `jax<0.5` pin downgrade, then bouncing back to a buggy 0.7.2 jaxlib
# whose mhlo dialect crashes on lazy import (Python 3.12 type-hint subscript bug).
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.isdir('/content/StockSurvey'):
        !git clone -q --depth 1 --no-recurse-submodules https://github.com/sughodke/StockSurvey.git /content/StockSurvey
    !pip install -q --no-deps /content/StockSurvey/packages/wavelets
    !pip install -q --no-deps /content/StockSurvey/packages/loaders
    !pip install -q --no-deps /content/StockSurvey/packages/indicators
    !pip install -q yfinance  # ss_loaders Yahoo fallback path
    print('setup done')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax.training import train_state

from ss_loaders import load_price_matrix
from ss_wavelets import causal_cwt
from ss_indicators import rsi, macd, bbands

print('jax:', jax.__version__, '| devices:', jax.devices())

## Config

Keep the dataset small for the workbook. Bump `max_tickers` and epochs once the end-to-end wiring works.

In [ ]:
CONFIG = {
    'data_dir': './Nasdaq3347',          # used when present (local repo workflow)
    'fallback_tickers': [                # used in Colab / when data_dir missing
        'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',
        'NVDA', 'TSLA', 'NFLX', 'JPM',   'V',
        'UNH',  'HD',   'BAC',  'PG',    'KO',
    ],
    'min_history': 504,
    'start_date':  '2018-01-01',
    'end_date':    '2024-12-31',
    'max_tickers': 15,
    # 8 scales: backbone has 3 ConvBlocks with stride-2 on the scale axis (8 -> 4 -> 2 -> 1).
    'scales':            [3, 7, 12, 21, 42, 63, 90, 126],
    'cwt_norm_lookback': 120,
    'window_len':        128,
    'step':              32,
    'train_frac':        0.7,
    'seed':              42,
    'lr':                1e-3,
    'batch_size':        32,
    'epochs':            5,
}
print({k: v for k, v in CONFIG.items() if k != 'fallback_tickers'})


## 1. Data + CWT (deterministic encoder)

Load prices via `ss_loaders.load_price_matrix` and run the causal CWT via `ss_wavelets.causal_cwt` once over the full date range; later we slice windows out of it.

In [ ]:
# Load prices: prefer the local Kaggle dump; fall back to Yahoo for a small
# Colab-friendly cohort.
import os
from datetime import datetime

if os.path.isdir(CONFIG['data_dir']):
    prices, _, _ = load_price_matrix(
        CONFIG['data_dir'],
        min_history=CONFIG['min_history'],
        start_date=CONFIG['start_date'],
        end_date=CONFIG['end_date'])
    keep_tickers = sorted(prices.columns)[:CONFIG['max_tickers']]
    prices = prices[keep_tickers]
else:
    from ss_loaders import load_yahoo
    start = datetime.strptime(CONFIG['start_date'], '%Y-%m-%d')
    end   = datetime.strptime(CONFIG['end_date'],   '%Y-%m-%d')
    cols = {}
    for t in CONFIG['fallback_tickers'][:CONFIG['max_tickers']]:
        try:
            df = load_yahoo(start, end, t)
            cols[t] = df['close']
        except Exception as e:
            print(f'  skip {t}: {e}')
    prices = pd.concat(cols, axis=1).dropna()

print('Using', prices.shape[1], 'tickers,', prices.shape[0], 'dates')


In [ ]:
price_arr = prices.values  # (n_dates, n_tickers)

print('Computing causal CWT (shape n_scales × n_dates × n_tickers)...')
cwt = causal_cwt(price_arr,
                        scales=CONFIG['scales'],
                        lookback=CONFIG['cwt_norm_lookback'])
print('CWT shape:', cwt.shape)

In [ ]:
# Indicators come from `ss_indicators` (JAX). Cast back to numpy for downstream slicing.
rsi_arr = np.asarray(rsi(price_arr, n=7))                          # (T, N) in [0, 100]
macd_line, macd_sig, macd_hist = (np.asarray(a) for a in macd(price_arr))
bb_mid, bb_up, bb_lo = (np.asarray(a) for a in bbands(price_arr))

# Normalize so every channel is dimensionless and ~O(0.01-1)
tgt_rsi  = rsi_arr / 100.0                                          # (T, N)
tgt_macd = np.stack([macd_line, macd_sig, macd_hist],
                    axis=-1) / price_arr[..., None]                 # (T, N, 3)
tgt_bb   = (np.stack([bb_up, bb_mid, bb_lo], axis=-1)
            - price_arr[..., None]) / price_arr[..., None]          # (T, N, 3)

# Assemble into one target tensor: (T, N, 7) — order [rsi, macd_line, macd_sig, macd_hist, bb_up, bb_mid, bb_lo]
targets = np.concatenate([tgt_rsi[..., None], tgt_macd, tgt_bb], axis=-1)
HEAD_NAMES  = ['rsi', 'macd_line', 'macd_signal', 'macd_hist', 'bb_upper', 'bb_middle', 'bb_lower']
HEAD_CHANNELS = {'rsi': 1, 'macd': 3, 'bbands': 3}
HEAD_SLICES  = {'rsi': slice(0, 1), 'macd': slice(1, 4), 'bbands': slice(4, 7)}

print('targets shape:', targets.shape, '(n_dates, n_tickers, 7)')
print('per-channel means:', np.nanmean(targets.reshape(-1, 7), axis=0).round(4))
print('per-channel stds :', np.nanstd(targets.reshape(-1, 7),  axis=0).round(4))

## 3. Window sampler

Split **by ticker** — train tickers are entirely disjoint from val tickers. Each sample is one `(ticker, end_date)` pair; we slice the last `W` steps of the CWT tensor and the target tensor at that ticker/date.

In [ ]:
rng = np.random.default_rng(CONFIG['seed'])
n_tickers = price_arr.shape[1]
perm = rng.permutation(n_tickers)
n_train = int(CONFIG['train_frac'] * n_tickers)
train_ticker_idx = np.sort(perm[:n_train])
val_ticker_idx   = np.sort(perm[n_train:])
print('train tickers:', len(train_ticker_idx), '| val tickers:', len(val_ticker_idx))

W = CONFIG['window_len']
step = CONFIG['step']
min_start = max(CONFIG['cwt_norm_lookback'], W)  # ensure CWT is warm and window fits
end_dates = np.arange(min_start + W - 1, price_arr.shape[0], step)
print('end-date anchors:', len(end_dates))


def build_windows(ticker_idx, end_idx):
    """Materialize (cwt_windows, target_windows) for given tickers and end-date anchors.

    cwt tensor layout in backtest_bt: (n_scales, n_dates, n_tickers)
    Per sample: (n_scales, W, 1) — NHWC with H=scales, W=time, C=1.
    Target per sample: (W, 7).
    """
    n_scales = cwt.shape[0]
    X = np.empty((len(ticker_idx) * len(end_idx), n_scales, W, 1), dtype=np.float32)
    Y = np.empty((len(ticker_idx) * len(end_idx), W, targets.shape[-1]), dtype=np.float32)
    k = 0
    for ti in ticker_idx:
        for ed in end_idx:
            start = ed - W + 1
            X[k, :, :, 0] = cwt[:, start:ed + 1, ti]
            Y[k] = targets[start:ed + 1, ti]
            k += 1
    return X, Y


X_train, Y_train = build_windows(train_ticker_idx, end_dates)
X_val,   Y_val   = build_windows(val_ticker_idx,   end_dates)

# Drop any windows with NaN in target (early BB rows) or CWT
def clean(X, Y):
    mask = np.isfinite(X).all(axis=(1, 2, 3)) & np.isfinite(Y).all(axis=(1, 2))
    return X[mask], Y[mask]

X_train, Y_train = clean(X_train, Y_train)
X_val,   Y_val   = clean(X_val,   Y_val)
print('train:', X_train.shape, Y_train.shape)
print('val  :', X_val.shape,   Y_val.shape)

## 4. Model: small 2D CNN backbone + multi-head decoder

**Backbone.** Three 2D conv blocks that downsample along the *scale* dimension only, preserving the time axis. After the last block the scale dim is collapsed to 1, leaving a per-timestep feature vector of shape `(batch, W, C)`.

**Heads.** One `Dense` layer per indicator family, applied per timestep. Full-trajectory prediction falls out naturally because the time dimension is preserved.

**Returns-head stub.** Wired in but disabled — flip `predict_returns=True` later to jointly train on forward-return targets.

In [ ]:
class ConvBlock(nn.Module):
    features: int
    scale_stride: int = 2  # downsample scale dim only

    @nn.compact
    def __call__(self, x):
        x = nn.Conv(self.features, kernel_size=(3, 3), strides=(self.scale_stride, 1),
                    padding='SAME')(x)
        x = nn.GroupNorm(num_groups=8)(x)
        x = nn.gelu(x)
        x = nn.Conv(self.features, kernel_size=(3, 3), strides=(1, 1), padding='SAME')(x)
        x = nn.GroupNorm(num_groups=8)(x)
        x = nn.gelu(x)
        return x


class CWTVisionMultiHead(nn.Module):
    latent_c: int = 128
    predict_returns: bool = False  # stub — flip on for multi-task predictive training

    @nn.compact
    def __call__(self, x):
        # x: (B, n_scales=8, W=128, 1) in NHWC
        x = ConvBlock(features=32)(x)   # (B, 4, W, 32)
        x = ConvBlock(features=64)(x)   # (B, 2, W, 64)
        x = ConvBlock(features=self.latent_c)(x)  # (B, 1, W, 128)
        x = jnp.squeeze(x, axis=1)      # (B, W, 128)

        # Per-timestep heads (Dense applies to the last axis)
        out = {
            'rsi':    nn.Dense(1)(x),   # (B, W, 1)
            'macd':   nn.Dense(3)(x),   # (B, W, 3)
            'bbands': nn.Dense(3)(x),   # (B, W, 3)
        }
        if self.predict_returns:
            # TODO: activate for multi-task predictive training
            out['returns'] = nn.Dense(1)(x)  # (B, W, 1)
        return out


model = CWTVisionMultiHead(predict_returns=False)
rng_key = jax.random.PRNGKey(CONFIG['seed'])
dummy = jnp.zeros((2, X_train.shape[1], W, 1), dtype=jnp.float32)
params = model.init(rng_key, dummy)['params']
print('param count:', sum(p.size for p in jax.tree_util.tree_leaves(params)))

## 5. Training loop

Per-head MSE summed for the total loss; individual head losses logged every epoch so you can see which family the model learns fastest.

In [ ]:
def split_targets(y):
    """Split the (B, W, 7) target into a dict keyed by head name."""
    return {
        'rsi':    y[..., HEAD_SLICES['rsi']],
        'macd':   y[..., HEAD_SLICES['macd']],
        'bbands': y[..., HEAD_SLICES['bbands']],
    }


def loss_fn(params, x, y):
    pred = model.apply({'params': params}, x)
    tgt = split_targets(y)
    losses = {k: jnp.mean((pred[k] - tgt[k]) ** 2) for k in ['rsi', 'macd', 'bbands']}
    total = losses['rsi'] + losses['macd'] + losses['bbands']
    return total, losses


optimizer = optax.adam(CONFIG['lr'])
state = train_state.TrainState.create(apply_fn=model.apply, params=params, tx=optimizer)


@jax.jit
def train_step(state, x, y):
    (total, losses), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params, x, y)
    state = state.apply_gradients(grads=grads)
    return state, total, losses


@jax.jit
def eval_step(params, x, y):
    total, losses = loss_fn(params, x, y)
    return total, losses


def iterate_batches(X, Y, batch_size, shuffle_rng=None):
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle_rng is not None:
        shuffle_rng.shuffle(idx)
    for start in range(0, n, batch_size):
        sel = idx[start:start + batch_size]
        yield jnp.asarray(X[sel]), jnp.asarray(Y[sel])

In [ ]:
history = {'train_total': [], 'val_total': [], 'train_heads': [], 'val_heads': []}
shuffle_rng = np.random.default_rng(CONFIG['seed'])

for epoch in range(CONFIG['epochs']):
    # --- train
    tr_totals, tr_heads = [], {'rsi': [], 'macd': [], 'bbands': []}
    for xb, yb in iterate_batches(X_train, Y_train, CONFIG['batch_size'], shuffle_rng):
        state, total, losses = train_step(state, xb, yb)
        tr_totals.append(float(total))
        for k, v in losses.items():
            tr_heads[k].append(float(v))

    # --- val
    va_totals, va_heads = [], {'rsi': [], 'macd': [], 'bbands': []}
    for xb, yb in iterate_batches(X_val, Y_val, CONFIG['batch_size']):
        total, losses = eval_step(state.params, xb, yb)
        va_totals.append(float(total))
        for k, v in losses.items():
            va_heads[k].append(float(v))

    history['train_total'].append(np.mean(tr_totals))
    history['val_total'].append(np.mean(va_totals))
    history['train_heads'].append({k: np.mean(v) for k, v in tr_heads.items()})
    history['val_heads'].append({k: np.mean(v) for k, v in va_heads.items()})

    th = history['train_heads'][-1]
    vh = history['val_heads'][-1]
    print(f'epoch {epoch:02d}  '
          f'train total={history["train_total"][-1]:.5f}  '
          f'[rsi {th["rsi"]:.5f} macd {th["macd"]:.5f} bb {th["bbands"]:.5f}]  '
          f'| val total={history["val_total"][-1]:.5f}  '
          f'[rsi {vh["rsi"]:.5f} macd {vh["macd"]:.5f} bb {vh["bbands"]:.5f}]')

## 6. Evaluation + sanity plots

Plot true vs. predicted indicator trajectories for a random **held-out** window.

In [ ]:
# Loss curves
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history['train_total'], label='train')
ax[0].plot(history['val_total'],   label='val')
ax[0].set_title('total MSE')
ax[0].set_xlabel('epoch'); ax[0].legend()

for head in ['rsi', 'macd', 'bbands']:
    ax[1].plot([h[head] for h in history['val_heads']], label=f'val/{head}')
ax[1].set_title('per-head val MSE')
ax[1].set_xlabel('epoch'); ax[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Pick a random val window and plot true vs. predicted for each indicator family
pick_rng = np.random.default_rng(CONFIG['seed'] + 1)
idx = pick_rng.integers(0, X_val.shape[0])

xb = jnp.asarray(X_val[idx:idx + 1])
yb = Y_val[idx]  # (W, 7)
pred = model.apply({'params': state.params}, xb)
pred = {k: np.asarray(v[0]) for k, v in pred.items()}  # drop batch
tgt  = {
    'rsi':    yb[:, HEAD_SLICES['rsi']],
    'macd':   yb[:, HEAD_SLICES['macd']],
    'bbands': yb[:, HEAD_SLICES['bbands']],
}

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
t = np.arange(W)

axes[0].plot(t, tgt['rsi'][:, 0] * 100, label='true RSI', lw=1.5)
axes[0].plot(t, pred['rsi'][:, 0] * 100, '--', label='pred RSI', lw=1.5)
axes[0].set_ylabel('RSI'); axes[0].legend(); axes[0].grid(alpha=0.3)

for i, name in enumerate(['line', 'signal', 'hist']):
    axes[1].plot(t, tgt['macd'][:, i],       label=f'true {name}', lw=1.2)
    axes[1].plot(t, pred['macd'][:, i], '--', label=f'pred {name}', lw=1.2)
axes[1].set_ylabel('MACD / close'); axes[1].legend(ncol=2); axes[1].grid(alpha=0.3)

for i, name in enumerate(['upper', 'middle', 'lower']):
    axes[2].plot(t, tgt['bbands'][:, i],       label=f'true {name}', lw=1.2)
    axes[2].plot(t, pred['bbands'][:, i], '--', label=f'pred {name}', lw=1.2)
axes[2].set_ylabel('(BB - close) / close'); axes[2].legend(ncol=2); axes[2].grid(alpha=0.3)
axes[2].set_xlabel('step in window')

plt.suptitle(f'val window #{idx}  (held-out ticker)')
plt.tight_layout(); plt.show()

## Sanity baseline: linear probe

Before scaling up, ask whether the CNN architecture is actually earning its keep. Flatten each `(n_scales, W)` CWT window into a single feature vector and fit `Ridge` per indicator head — same input information the CNN sees, no spatial inductive bias.

If Ridge matches or beats the CNN's val MSE, the 2D conv structure isn't doing useful work on this task and any further experiments (predictive heads, returns head) should reconsider the architecture first.

In [ ]:
from sklearn.linear_model import Ridge

# Flatten each window: (N, n_scales, W, 1) -> (N, n_scales*W)
X_tr_flat = X_train.reshape(X_train.shape[0], -1)
X_va_flat = X_val.reshape(X_val.shape[0], -1)
print(f'probe input dim: {X_tr_flat.shape[1]}  |  train samples: {X_tr_flat.shape[0]}')
print()

cnn_val_final = history['val_heads'][-1]

print(f'{"head":7s}  {"best_alpha":>10s}  {"ridge_val":>10s}  {"cnn_val":>10s}  {"ratio":>7s}  verdict')
print('-' * 65)
for head_name, slc in HEAD_SLICES.items():
    Y_tr = Y_train[:, :, slc].reshape(Y_train.shape[0], -1)
    Y_va = Y_val[:,   :, slc].reshape(Y_val.shape[0],   -1)

    # 1024-d input vs ~480 train samples is heavily underdetermined; sweep alpha.
    best_alpha, best_val = None, float('inf')
    for alpha in [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]:
        reg = Ridge(alpha=alpha).fit(X_tr_flat, Y_tr)
        mse_va = float(((reg.predict(X_va_flat) - Y_va) ** 2).mean())
        if mse_va < best_val:
            best_alpha, best_val = alpha, mse_va

    cnn_va = cnn_val_final[head_name]
    ratio  = best_val / cnn_va
    verdict = 'Ridge wins (CNN suspect)' if ratio < 0.9 else (
              'tied'                     if ratio < 1.1 else
              'CNN earns its keep')
    print(f'{head_name:7s}  {best_alpha:10.2f}  {best_val:10.5f}  {cnn_va:10.5f}  {ratio:6.2f}x  {verdict}')

## 7. Next steps

**TODO — predictive RSI head + backtest with ground truth.**
Replace the RSI *reconstruction* target with *next-step* (or k-step-ahead) RSI. Train the same architecture but the loss compares the model's RSI prediction at time `t` against the true RSI at time `t + k`. Then plug predicted RSI into the existing ranking/decider logic (`models/directors.py::NumpyDecider`) and run it through `backtest_bt.py` against the classical-RSI baseline. The interesting question is whether a CWT-conditioned RSI predictor earns a Sharpe edge over "use current RSI directly."

**TODO — activate the returns head.**
Flip `predict_returns=True` on the model and extend `split_targets` to carry forward-return labels (e.g., `log(close[t+20] / close[t])`). Sum its MSE into the total loss. The indicator heads then act as **auxiliary losses** anchoring the shared latent in TA-relevant structure while the returns head is the actual predictive target. Compare val Sharpe on the returns head with vs. without the indicator aux losses — this is the only version of this experiment where the "does the vision backbone help?" question has a clean answer.

**TODO — sanity baseline.**
Before spending compute scaling this up, run a linear probe: flatten `(n_scales × W)` CWT coefficients, fit `sklearn.linear_model.Ridge` per-head. If the linear probe matches the CNN on indicator reconstruction, the 2D architecture isn't earning its keep on this task and you should revisit the design before adding the predictive head.